# RoIAlign (and RoIPool) from Scratch — Solution

**Difficulty**: 🔴 Hard

### Problem Statement

A two-stage detector (Faster R-CNN, Mask R-CNN, Cascade R-CNN) first proposes boxes and then runs a small head on a **fixed-size feature crop for every box**. torchvision's Faster R-CNN pools a 7×7 crop per box for the box head, and its Mask R-CNN and Keypoint R-CNN pool 14×14 crops for the mask and keypoint heads (`MultiScaleRoIAlign(..., sampling_ratio=2)`). Boxes are given in image pixels, but the features come from a map with stride 4–32, so the crop has to be read at fractional positions of a coarse grid.

**RoIPool** (Fast R-CNN, Girshick 2015) does this with two roundings: it snaps the box to the feature grid, splits it into integer-edged bins and max-pools each bin. At stride 16 the first rounding alone can move the crop by up to 8 image pixels. That hardly matters for classification, but it does matter for a 28×28 mask or a keypoint heatmap predicted in the crop's coordinates. **RoIAlign** (Mask R-CNN, He et al. 2017) removes every rounding: it takes bilinear samples at regular points inside each bin and averages them. In Mask R-CNN's ablation this change alone raised mask AP from 26.9 to 30.3 with stride-16 features (Table 2c), raised it by 7.3 AP with stride-32 features (Table 2d), and raised keypoint AP by 4.4 (Table 6).

Implement both, matching `torchvision.ops.roi_align` and `torchvision.ops.roi_pool` **exactly**, including their border and rounding conventions. You already wrote the clamped two-tap bilinear gather in **bilinear-interpolation**. This problem is about the geometry around that gather: where the samples go, which ones count, and how they are averaged.

Coordinates: `x` is the width (column) axis and `y` the height (row) axis. A box row is `[batch_index, x1, y1, x2, y2]` in **image** pixels, and `spatial_scale` $s$ maps it to the feature map (e.g. $s = 1/16$ for a stride-16 map).

**RoIAlign for one box.** The y axis is shown below; the x axis is identical with $W$ and $\text{out}_w$.

1. **Scale and shift, with no rounding:** $y_1' = y_1 s - o$ and $y_2' = y_2 s - o$, where $o = 0.5$ if `aligned` else $0$.
2. **Size:** $h = y_2' - y_1'$. Only when `aligned=False`, $h \leftarrow \max(h, 1)$ (torchvision forces malformed RoIs to 1×1).
3. **Bins and grid:** the bin size is $b = h / \text{out}_h$. The number of samples per bin along this axis is $g = $ `sampling_ratio` if it is $> 0$, else the **adaptive** $g = \lceil h / \text{out}_h \rceil$. It is computed per box and per axis, and it can be $0$ (a zero-height box with `aligned=True`).
4. **Sample positions:** sample $i$ of bin $p$ is at the centre of the $i$-th of $g$ equal sub-cells,
$$y_{p,i} = y_1' + p\,b + \left(i + \tfrac12\right)\frac{b}{g}, \qquad i = 0, \dots, g - 1.$$
5. **Reading a sample** at $(y, x)$ from $F = $ `input[batch_index]`. If $y < -1$, $y > H$, $x < -1$ or $x > W$, the sample is $0$. Otherwise clamp $y$ to $[0, H-1]$ and $x$ to $[0, W-1]$, then interpolate bilinearly between rows $\lfloor y \rfloor$ and $\min(\lfloor y \rfloor + 1, H - 1)$, and likewise for columns. So a sample less than one pixel outside the map reads the border pixel at **full** weight, and a sample further out is dropped but **still counts** in the denominator.
6. **Average:**
$$\text{out}[k, :, p, q] = \frac{1}{\max(g_h\, g_w,\ 1)} \sum_{i=0}^{g_h - 1} \sum_{j=0}^{g_w - 1} \text{sample}(y_{p,i},\ x_{q,j}).$$

`aligned=True` (Detectron2's "ROIAlignV2") uses the half-pixel pixel model from `align_corners=False`. Pixel $i$ covers $[i, i+1)$ and its centre is at $i + \tfrac12$, so a continuous box coordinate $c$ corresponds to index $c - \tfrac12$. With `aligned=False` (the legacy default, which torchvision's detection models still use) every sample sits half a pixel off.

**RoIPool for one box.**

1. Snap the box to the grid: $x_1^r = \operatorname{round}(x_1 s)$, and likewise for $y_1, x_2, y_2$. This is C's `round`, which sends halves **away from zero** ($2.5 \to 3$, $-0.5 \to -1$). `torch.round` and Python's `round` send halves to the nearest even integer instead.
2. Size: $w = \max(x_2^r - x_1^r + 1,\ 1)$, and the same for $h$. The `+1` is the legacy inclusive-end convention.
3. Bins: $b_h = h / \text{out}_h$ is a float. Bin $p$ covers rows $[\lfloor p\, b_h \rfloor,\ \lceil (p+1)\, b_h \rceil) + y_1^r$, and both ends are then clipped to $[0, H]$. Columns work the same way.
4. The output is the **max** over the bin's pixels, or $0$ if the clipped bin is empty.

---

### Requirements

1. **`roi_align(input, boxes, output_size, spatial_scale=1.0, sampling_ratio=-1, aligned=False) -> Tensor`**
   - `input`: `(N, C, H, W)`, float32 or float64. `boxes`: `(K, 5)` with the same dtype, rows `[batch_index, x1, y1, x2, y2]` in image coordinates with $x_1 \le x_2$, $y_1 \le y_2$. Boxes may stick out of the image or lie completely outside it. `output_size = (out_h, out_w)`.
   - Return `(K, C, out_h, out_w)` with `input`'s dtype. With `K = 0`, return an empty `(0, C, out_h, out_w)` tensor.
   - Match `torchvision.ops.roi_align` to `1e-8` in float64 (`2e-4` in float32) for both values of `aligned`, fixed and adaptive `sampling_ratio`, any `spatial_scale`, several images per batch, and tiny, zero-size, partly-outside and fully-outside boxes.
   - Never return NaN. An empty sampling grid gives $0$.
   - Differentiable with respect to `input`, with the same gradient as torchvision: each input pixel collects `weight × upstream / count` from every sample of every box that reads it. No gradient with respect to `boxes` is required.
   - Vectorized: a Python loop over the K boxes is fine, because the adaptive grid size differs per box. Do **not** loop over channels, bins, sampling points or pixels.
2. **`roi_pool(input, boxes, output_size, spatial_scale=1.0) -> Tensor`** has the same arguments and shapes and must match `torchvision.ops.roi_pool` exactly. Loops over boxes and bins are fine; vectorize over channels.
   - Floating-point subtlety: torchvision computes $\lfloor p \cdot (h/\text{out}) \rfloor$ and $\lceil (p+1)(h/\text{out}) \rceil$ in floating point. For a few (size, out) pairs this differs from exact integer arithmetic, e.g. $\lceil 7 \cdot (29/7) \rceil = 30$ in float64. The grader avoids those pairs, so either style passes.

---

### Constraints

- ✅ Pure PyTorch: `torch.arange`, `floor`, `clamp`, advanced indexing, `gather`, `index_select`, `einsum`, `scatter_add_`.
- ✅ `F.grid_sample` is allowed for the bilinear step, but only if you reproduce the clamp-then-cut-off border rule and convert pixel coordinates to its normalised coordinates with the formula that matches its `align_corners` flag.
- ✅ Converting the **box** coordinates to Python floats (`boxes.tolist()`) is fine. The gradient only has to flow to `input`.
- ❌ Do not call `torchvision.ops` (only the optional cross-check cell imports it).
- ❌ No Python loop over channels, bins or sampling points in `roi_align`. The grader counts torch calls for the same 3 boxes at C=1 / 2×2 output / 1 sample per bin and at C=6 / 7×7 / 3×3 samples, and the two counts must be about the same.
- ❌ No `.item()`, `.tolist()` or numpy on the **feature values** in `roi_align` (the gradient has to flow).

---

<details>
  <summary>💡 Hint</summary>

  - Draw one axis: a box from $y_1'$ to $y_2'$ split into `out_h` bins, each with `g` samples at the centres of `g` equal sub-cells. Where is sample 0 of bin 0? Where is the last sample of the last bin?
  - The samples of one box form a tensor product: every y position is paired with every x position. Build the y positions as one vector of length `out_h * g_h` and the x positions as one vector of length `out_w * g_w`, then reuse your separable two-tap gather from bilinear-interpolation. What shape do you get before averaging, and which axes do you sum?
  - How can an out-of-range sample contribute 0 without breaking the gather? Think about its weights, not its indices.
  - What does `sampling_ratio=-1` give for a zero-width box with `aligned=True`, and what do you divide by then?
  - For `roi_pool`, try `torch.round(torch.tensor([0.5, 1.5, 2.5]))` and compare it with the rule above.

</details>

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
torch.manual_seed(0)
torch.set_printoptions(precision=3, linewidth=120)

# A linear "ramp" feature map: value = 10 * y + x at pixel (y, x). Bilinear interpolation of a
# linear function is exact, and the samples of a bin are symmetric about the bin centre, so for a
# box whose samples all land inside the map, RoIAlign returns the ramp at each bin centre.
H, W = 8, 10
ramp = (10 * torch.arange(H, dtype=torch.float64).view(H, 1)
        + torch.arange(W, dtype=torch.float64).view(1, W)).view(1, 1, H, W)
print(ramp[0, 0])

# A batch of 2 stride-4 feature maps for a 48 x 64 image, and some boxes in image pixels.
feats = torch.randn(2, 4, 12, 16, dtype=torch.float64)
boxes = torch.tensor([[0, 4.0, 6.0, 30.0, 40.0],     # [batch_index, x1, y1, x2, y2]
                      [1, 10.0, 2.0, 60.0, 20.0],
                      [0, -8.0, 30.0, 20.0, 56.0]],  # sticks out of the left and bottom edges
                     dtype=torch.float64)

In [ ]:
def _sample_axis(start, length, n_bins, grid, size, dtype, device):
    """
    Sample positions along one axis of one box, as two bilinear taps per sample.

    Args:
        start: box start on this axis, in feature-map coordinates (already scaled and shifted).
        length: box length on this axis (already clamped to >= 1 when aligned=False).
        n_bins: number of output bins on this axis (out_h or out_w).
        grid: samples per bin on this axis (>= 0).
        size: feature-map size on this axis (H or W).

    Returns:
        lo, hi: (n_bins * grid,) int64 tap indices, 0 <= lo <= hi <= size - 1.
        w_lo, w_hi: (n_bins * grid,) tap weights; both 0 for a sample beyond the cut-off.
        Samples are ordered bin-major: entry p * grid + i is sample i of bin p.
    """
    bin_size = length / n_bins
    p = torch.arange(n_bins, dtype=dtype, device=device).view(-1, 1)    # (n_bins, 1)
    i = torch.arange(grid, dtype=dtype, device=device).view(1, -1)      # (1, grid)
    s = (start + p * bin_size + (i + 0.5) * bin_size / grid).reshape(-1)
    # torchvision drops a sample only beyond one pixel outside: y < -1 or y > size.
    inside = ((s >= -1.0) & (s <= size)).to(dtype)
    # Everything else is clamped onto the map, so -1 <= y < 0 reads row 0 at full weight.
    s = s.clamp(min=0.0, max=size - 1)
    lo = s.floor().long()
    hi = (lo + 1).clamp(max=size - 1)
    frac = s - lo
    return lo, hi, (1 - frac) * inside, frac * inside


def roi_align(input, boxes, output_size, spatial_scale=1.0, sampling_ratio=-1, aligned=False):
    """
    RoIAlign with torchvision.ops.roi_align semantics.

    Args:
        input: (N, C, H, W) feature map.
        boxes: (K, 5) rows [batch_index, x1, y1, x2, y2] in input-image coordinates.
        output_size: (out_h, out_w).
        spatial_scale: feature-map size / image size (e.g. 1/16 for a stride-16 map).
        sampling_ratio: samples per bin per axis; <= 0 means adaptive ceil(roi / out).
        aligned: subtract 0.5 after scaling (half-pixel pixel model, Detectron2).

    Returns:
        (K, C, out_h, out_w) tensor with input's dtype, differentiable w.r.t. input.
    """
    N, C, H, W = input.shape
    out_h, out_w = output_size
    offset = 0.5 if aligned else 0.0
    outs = []
    for b, x1, y1, x2, y2 in boxes.tolist():           # the only Python loop: over boxes
        # 1) box -> continuous feature-map coordinates; no rounding anywhere.
        x1, y1 = x1 * spatial_scale - offset, y1 * spatial_scale - offset
        x2, y2 = x2 * spatial_scale - offset, y2 * spatial_scale - offset
        roi_w, roi_h = x2 - x1, y2 - y1
        if not aligned:                                 # legacy: force malformed RoIs to 1x1
            roi_w, roi_h = max(roi_w, 1.0), max(roi_h, 1.0)
        # 2) samples per bin: fixed, or adaptive (about one per feature-map pixel).
        gh = sampling_ratio if sampling_ratio > 0 else math.ceil(roi_h / out_h)
        gw = sampling_ratio if sampling_ratio > 0 else math.ceil(roi_w / out_w)
        ylo, yhi, wy_lo, wy_hi = _sample_axis(y1, roi_h, out_h, gh, H, input.dtype, input.device)
        xlo, xhi, wx_lo, wx_hi = _sample_axis(x1, roi_w, out_w, gw, W, input.dtype, input.device)
        # 3) separable bilinear gather: rows first, then columns (as in bilinear-interpolation).
        img = input[int(b)]                                                     # (C, H, W)
        rows = img[:, ylo, :] * wy_lo[:, None] + img[:, yhi, :] * wy_hi[:, None]  # (C, out_h*gh, W)
        vals = rows[:, :, xlo] * wx_lo + rows[:, :, xhi] * wx_hi                  # (C, out_h*gh, out_w*gw)
        # 4) average the gh x gw samples of each bin; an empty grid gives 0, not 0/0.
        vals = vals.view(C, out_h, gh, out_w, gw).sum(dim=(2, 4))
        outs.append(vals / max(gh * gw, 1))
    if not outs:
        return input.new_zeros(0, C, out_h, out_w)
    return torch.stack(outs)

In [ ]:
# Test 1: shapes, dtype, no boxes
out = roi_align(feats, boxes, (7, 7), spatial_scale=0.25)
assert out.shape == (3, 4, 7, 7), f"expected (3, 4, 7, 7), got {tuple(out.shape)}"
assert out.dtype == torch.float64
assert roi_align(feats.float(), boxes.float(), (2, 3), spatial_scale=0.25).dtype == torch.float32
empty = roi_align(feats, torch.zeros(0, 5, dtype=torch.float64), (7, 7))
assert empty.shape == (0, 4, 7, 7), f"no boxes -> (0, C, 7, 7), got {tuple(empty.shape)}"
print("Test 1 passed: shapes and dtypes")

In [ ]:
# Test 2: on the linear ramp, each bin equals the ramp at its centre
# Box x in [2, 6], y in [1, 5], 2x2 bins of size 2: bin centres y in {2, 4}, x in {3, 5}.
box = torch.tensor([[0, 2.0, 1.0, 6.0, 5.0]], dtype=torch.float64)
for sr in (1, 2, 3, -1):
    got = roi_align(ramp, box, (2, 2), sampling_ratio=sr, aligned=False)[0, 0]
    assert torch.allclose(got, torch.tensor([[23.0, 25.0], [43.0, 45.0]], dtype=torch.float64)), \
        f"aligned=False, sampling_ratio={sr}: got\n{got}"
    # aligned=True shifts every sample by -0.5 pixel in y and x: 10 * (-0.5) + (-0.5) = -5.5
    got = roi_align(ramp, box, (2, 2), sampling_ratio=sr, aligned=True)[0, 0]
    assert torch.allclose(got, torch.tensor([[17.5, 19.5], [37.5, 39.5]], dtype=torch.float64)), \
        f"aligned=True, sampling_ratio={sr}: got\n{got}"
# spatial_scale: the same box in a 2x larger image
got = roi_align(ramp, box * torch.tensor([1, 2, 2, 2, 2]), (2, 2), spatial_scale=0.5)[0, 0]
assert torch.allclose(got, torch.tensor([[23.0, 25.0], [43.0, 45.0]], dtype=torch.float64))
print("Test 2 passed: sample positions, aligned shift, spatial_scale")

In [ ]:
# Test 3: the border rule. A 1x1-pixel box with sampling_ratio=1 samples its centre.
def centre(yc, xc, img=ramp):
    b = torch.tensor([[0, xc - 0.5, yc - 0.5, xc + 0.5, yc + 0.5]], dtype=torch.float64)
    return roi_align(img, b, (1, 1), sampling_ratio=1).item()

assert centre(-0.5, 3.0) == 3.0, "-1 <= y < 0 reads row 0 at full weight (clamp, no zero padding)"
assert centre(-1.0, 3.0) == 3.0, "y = -1 exactly is still clamped to row 0"
assert centre(-1.5, 3.0) == 0.0, "y < -1 contributes 0"
assert centre(8.0, 3.0) == 73.0, "y = H exactly reads row H - 1"
assert centre(8.5, 3.0) == 0.0, "y > H contributes 0"
assert centre(2.0, -0.75) == 20.0 and centre(2.0, 10.25) == 0.0, "same rule along x"
# A box entirely outside the map gives zeros (not NaN).
far = torch.tensor([[0, 30.0, 30.0, 40.0, 40.0]], dtype=torch.float64)
assert roi_align(ramp, far, (2, 2)).abs().max().item() == 0.0
# aligned=True with a zero-width box and an adaptive grid: ceil(0 / out) = 0 samples -> 0
flat = torch.tensor([[0, 3.0, 1.0, 3.0, 6.0]], dtype=torch.float64)
z = roi_align(ramp, flat, (2, 2), sampling_ratio=-1, aligned=True)
assert torch.isfinite(z).all() and z.abs().max().item() == 0.0, f"expected zeros, got {z}"
print("Test 3 passed: border rule and empty grids")

In [ ]:
# Test 4: adaptive grid, batch index
# This box is 5 x 5.5 feature pixels -> with a 2x2 output the adaptive grid is ceil(2.5) x ceil(2.75) = 3 x 3.
b = torch.tensor([[1, 1.0, 2.0, 6.5, 7.0]], dtype=torch.float64)
adaptive = roi_align(feats, b, (2, 2), sampling_ratio=-1)
assert torch.allclose(adaptive, roi_align(feats, b, (2, 2), sampling_ratio=3)), \
    "sampling_ratio=-1 should use ceil(roi / out) = 3 samples per axis here"
assert not torch.allclose(adaptive, roi_align(feats, b, (2, 2), sampling_ratio=2)), \
    "adaptive must not be a fixed 2x2 grid"
assert torch.allclose(roi_align(feats, b, (2, 2), sampling_ratio=0), adaptive), "0 also means adaptive"
# batch index 1 reads feats[1]
b0 = b.clone(); b0[0, 0] = 0
assert torch.allclose(adaptive, roi_align(feats[1:], b0, (2, 2))), "boxes[:, 0] selects the image"
print("Test 4 passed: adaptive sampling grid and batch index")

In [ ]:
# Test 5: gradients. For boxes whose samples all land inside the map, every output bin's
# weights sum to 1, so with an upstream gradient of ones the input gradient sums to K*out_h*out_w
# in every channel.
x = feats.clone().requires_grad_(True)
inside = torch.tensor([[0, 4.0, 4.0, 40.0, 30.0], [1, 8.0, 10.0, 50.0, 40.0], [1, 8.0, 10.0, 50.0, 40.0]],
                      dtype=torch.float64)
roi_align(x, inside, (3, 3), spatial_scale=0.25, sampling_ratio=2, aligned=True).sum().backward()
assert x.grad is not None and torch.isfinite(x.grad).all(), "no gradient reached the input"
assert abs(x.grad.sum().item() - 3 * 3 * 3 * 4) < 1e-9, x.grad.sum().item()  # K * out_h * out_w * C
# adding a box that lies entirely outside the map does not change the gradient
far_box = torch.tensor([[0, 200.0, 200.0, 220.0, 220.0]], dtype=torch.float64)
x.grad = None
roi_align(x, torch.cat([inside[:1], far_box]), (2, 2), 0.25).sum().backward()
g_both = x.grad.clone()
x.grad = None
roi_align(x, inside[:1], (2, 2), 0.25).sum().backward()
assert torch.allclose(g_both, x.grad), "samples beyond the cut-off must not receive gradient"
print("Test 5 passed: gradients")

In [ ]:
def _c_round(v):
    """C's round(): halfway cases away from zero (torch.round and Python's round go to even)."""
    return int(math.copysign(math.floor(abs(v) + 0.5), v))


def roi_pool(input, boxes, output_size, spatial_scale=1.0):
    """
    RoIPool (Fast R-CNN) with torchvision.ops.roi_pool semantics.

    Args:
        input: (N, C, H, W) feature map.
        boxes: (K, 5) rows [batch_index, x1, y1, x2, y2] in input-image coordinates.
        output_size: (out_h, out_w).
        spatial_scale: feature-map size / image size.

    Returns:
        (K, C, out_h, out_w): max over each quantized bin, 0 for an empty bin.
    """
    N, C, H, W = input.shape
    out_h, out_w = output_size
    out = input.new_zeros(boxes.shape[0], C, out_h, out_w)
    for k, (b, x1, y1, x2, y2) in enumerate(boxes.tolist()):
        # Quantization 1: snap the box to the feature grid (inclusive ends: +1).
        xs, ys, xe, ye = (_c_round(v * spatial_scale) for v in (x1, y1, x2, y2))
        roi_w, roi_h = max(xe - xs + 1, 1), max(ye - ys + 1, 1)
        bin_h, bin_w = roi_h / out_h, roi_w / out_w
        for ph in range(out_h):
            # Quantization 2: integer bin edges, then clip to the map.
            hs = min(max(math.floor(ph * bin_h) + ys, 0), H)
            he = min(max(math.ceil((ph + 1) * bin_h) + ys, 0), H)
            for pw in range(out_w):
                ws = min(max(math.floor(pw * bin_w) + xs, 0), W)
                we = min(max(math.ceil((pw + 1) * bin_w) + xs, 0), W)
                if he > hs and we > ws:                 # empty bin stays 0
                    out[k, :, ph, pw] = input[int(b), :, hs:he, ws:we].amax(dim=(1, 2))
    return out

In [ ]:
# Test 6: RoIPool by hand on a 6x7 map whose value is 7 * y + x
img = torch.arange(42, dtype=torch.float64).view(1, 1, 6, 7)
# Box (x 1..5, y 1..4): roi_w = 5 - 1 + 1 = 5, roi_h = 4 - 1 + 1 = 4; 2x2 bins of 2 x 2.5 pixels:
# rows [1, 3) and [3, 5); columns [1, 4) and [3, 6) (they overlap). Max = bottom-right of each.
got = roi_pool(img, torch.tensor([[0, 1.0, 1.0, 5.0, 4.0]], dtype=torch.float64), (2, 2))[0, 0]
assert torch.equal(got, torch.tensor([[17.0, 19.0], [31.0, 33.0]], dtype=torch.float64)), got
# Rounding: x1 = y1 = 1 and x2 = y2 = 5 at scale 0.5 are 0.5 and 2.5 -> C round gives 1 and 3.
got = roi_pool(img, torch.tensor([[0, 1.0, 1.0, 5.0, 5.0]], dtype=torch.float64), (1, 1), 0.5)
assert got.item() == 24.0, f"expected max over rows/cols 1..3 = 24 (round half away from zero), got {got.item()}"
# Empty bins (box off the map) are 0
off = roi_pool(img, torch.tensor([[0, 10.0, 10.0, 14.0, 12.0]], dtype=torch.float64), (2, 2))
assert off.abs().max().item() == 0.0, off
print("Test 6 passed: roi_pool")

In [ ]:
# Optional: cross-check against torchvision when it is installed (it is on Colab).
try:
    from torchvision.ops import roi_align as tv_roi_align, roi_pool as tv_roi_pool
except ImportError:
    print("torchvision not installed -- skipping the cross-check")
else:
    g = torch.Generator().manual_seed(1)
    fm = torch.randn(2, 3, 13, 17, generator=g, dtype=torch.float64)
    xy = torch.rand(40, 2, generator=g, dtype=torch.float64) * torch.tensor([90.0, 70.0]) - 10
    wh = torch.rand(40, 2, generator=g, dtype=torch.float64) * 40
    rois = torch.cat([torch.randint(0, 2, (40, 1), generator=g).double(), xy, xy + wh], 1)
    for sr in (-1, 2):
        for al in (False, True):
            assert torch.allclose(roi_align(fm, rois, (5, 4), 0.2, sr, al),
                                  tv_roi_align(fm, rois, (5, 4), 0.2, sr, al), atol=1e-8), (sr, al)
    q = rois.clone(); q[:, 1:] = (q[:, 1:] * 4).round() / 4      # quarter-pixel boxes
    assert torch.equal(roi_pool(fm, q, (3, 3), 0.25), tv_roi_pool(fm, q, (3, 3), 0.25))
    print("matches torchvision.ops.roi_align / roi_pool")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("roi-align", roi_align=roi_align, roi_pool=roi_pool)

In [ ]:
# Demo: why RoIPool hurts localisation. Slide a box right by quarter pixels on a stride-16 map
# and watch the pooled feature of a horizontal ramp. RoIPool's output is piecewise constant in
# the box position (it moves in jumps of a whole feature pixel); RoIAlign's moves smoothly.
ramp16 = torch.arange(32, dtype=torch.float64).view(1, 1, 1, 32).expand(1, 1, 8, 32).contiguous()
print(" shift(px)   RoIPool   RoIAlign(aligned=True)")
for shift in [0, 4, 8, 12, 16, 20, 24]:
    b = torch.tensor([[0, 64.0 + shift, 16.0, 160.0 + shift, 96.0]], dtype=torch.float64)
    p = roi_pool(ramp16, b, (1, 1), spatial_scale=1 / 16).item()
    a = roi_align(ramp16, b, (1, 1), spatial_scale=1 / 16, aligned=True).item()
    print(f"{shift:9d}   {p:7.3f}   {a:7.3f}")

### Why it works

- **No rounding anywhere.** Every quantity (box start, bin size, sample position) is a continuous function of the box, so the crop moves smoothly as the box moves (see the demo). Averaging $g_h \times g_w$ bilinear samples approximates the integral of the bilinearly interpolated feature over the bin, which is what Mask R-CNN wanted from a bin. RoIPool's output is piecewise constant in the box coordinates, so a sub-stride shift of the box does not change the crop at all, until it suddenly jumps by a whole feature pixel.
- **Separability.** The sample grid of a box is a tensor product $\{y_{p,i}\} \times \{x_{q,j}\}$. The bilinear weight of a tap factorises into a y-weight times an x-weight, and the cut-off mask factorises too ($y$ in range **and** $x$ in range). So the 2-D gather splits into a row gather followed by a column gather, which is the same separable gather as in bilinear-interpolation, applied to irregular coordinate vectors. Putting the mask into the weights, rather than the indices, keeps every index valid.
- **The border rule is the C kernel's.** `y <= 0 → 0`, then `y_low = (int) y`, and `y_low >= H-1 → y_low = y_high = H-1`. That is the same as clamping $y$ to $[0, H-1]$ and using taps $\lfloor y\rfloor$, $\min(\lfloor y\rfloor + 1, H-1)$. The only extra rule is the cut-off at $-1$ and $H$, which zeroes a sample but keeps it in the count.
- **`aligned=True`.** In the half-pixel model, pixel $i$ is centred at continuous coordinate $i + \tfrac12$, so a continuous box coordinate $c$ is at index $c - \tfrac12$. The legacy op skips this shift and samples half a feature pixel down and to the right of where the box says. The shift is applied after scaling because it is half a *feature* pixel, not half an image pixel.
- **Gradient.** The output is linear in `input`, with weights that depend only on the boxes. Autograd through advanced indexing applies the transpose (an `index_put_` with accumulation), and that is exactly torchvision's backward: each tap receives `grad × w / count`. Overlapping boxes accumulate.

### Complexity

- `roi_align`, per box: $O(C \cdot \text{out}_h g_h \cdot W)$ for the row pass and $O(C \cdot \text{out}_h g_h \cdot \text{out}_w g_w)$ for the column pass, plus $O(\text{out}_h g_h + \text{out}_w g_w)$ coordinate math. With adaptive sampling $g \approx \text{roi}/\text{out}$, so the number of samples is about the box area in feature pixels. Large boxes on a fine level are expensive, which is one reason FPN sends large boxes to coarse levels. A fixed `sampling_ratio` (torchvision's detection models use 2) bounds the cost per box.
- Memory: the `(C, out_h·g_h, W)` intermediate per box. torchvision's CPU kernel relies on the same fact, that taps and weights do not depend on the channel: it precomputes the four taps and weights of every sample of a box once (`pre_calc_for_bilinear_interpolate`) and reuses them for all channels.
- `roi_pool`: $O(C \cdot \text{box area})$ per box.

### Common bugs

- **Missing the −0.5 for `aligned=True`**, or subtracting it before scaling ($(x - 0.5)s$ instead of $xs - 0.5$).
- **Rounding the box or the bins.** That turns RoIAlign back into RoIPool.
- **Clamping out-of-range samples instead of dropping them**, so a box far outside the image copies the border. The opposite mistake is zero-padding samples in $[-1, 0)$, which is what `F.grid_sample(padding_mode="zeros")` does.
- **A fixed 2×2 grid when `sampling_ratio=-1`.** torchvision's default is adaptive, and `MultiScaleRoIAlign` in its detection models passes `sampling_ratio=2` explicitly.
- **Dividing by $g_h g_w = 0$** for zero-size boxes with `aligned=True` and an adaptive grid, which gives NaN.
- **Max instead of average.** Mask R-CNN reports the two as about equal (30.2 vs 30.3 mask AP, Table 2c), but torchvision averages.
- **Ignoring the batch index**, which silently crops every box from image 0.
- **`grid_sample` with mismatched normalisation**, e.g. $2x/(W-1) - 1$ while passing `align_corners=False`. That shifts every sample by up to half a pixel. With `align_corners=True`, $W = 1$ also divides by zero.
- **RoIPool:** `torch.round` (half to even) instead of C's half-away-from-zero, forgetting the `+1`, or returning `-inf` for empty bins.

### Interview follow-ups

1. **Why does RoIPool's quantization hurt masks and keypoints much more than classification?** Rounding the box to a stride-16 grid shifts the crop by up to 8 image pixels, and the integer bin edges add more jitter. A class label barely changes when the crop moves a few pixels, but a 28×28 mask or 56×56 keypoint heatmap is predicted in the crop's coordinate frame, so every predicted pixel moves with it. In Mask R-CNN, RoIAlign gave +3.4 mask AP over RoIPool at stride 16 (26.9 → 30.3 with averaging), +7.3 mask AP and +10.5 AP75 at stride 32, and +4.4 keypoint AP.
2. **Can you backpropagate into the box coordinates?** The bilinear weights are linear in the sample positions, so RoIAlign is differentiable almost everywhere in $(x_1, y_1, x_2, y_2)$. A pure-PyTorch version that keeps the boxes as tensors gets that gradient, except at the jumps (an adaptive grid changing size, a sample crossing the $-1$/$H$ cut-off). torchvision's op returns no gradient for the boxes. **PrRoI Pooling** (Precise RoI Pooling, from the IoU-Net paper by Jiang et al., 2018) integrates the interpolated feature exactly over each bin. It has no sampling-ratio hyperparameter and gives a continuous box gradient, which IoU-Net uses for optimisation-based box refinement. **Deformable RoI pooling** (Dai et al., 2017) learns a per-bin offset and gets its gradient through the same bilinear sampling.
3. **How does FPN decide which level a box is pooled from?** Eq. (1) of the FPN paper (Lin et al., 2017): $k = \lfloor k_0 + \log_2(\sqrt{wh}/224) \rfloor$ with $k_0 = 4$, clamped to the available levels (P2–P5), and that level's `spatial_scale` is $2^{-k}$. A 224×224 box (ImageNet pre-training size) goes to $P_4$, like the single-scale C4 detector, and halving the box side moves it one level finer. torchvision's `LevelMapper` adds $\epsilon = 10^{-6}$ inside the floor.
4. **How would you implement RoIAlign with `grid_sample`?** Build the `(out_h·g_h, out_w·g_w)` sample grid per box, convert to normalised coordinates with $(2x+1)/W - 1$ for `align_corners=False` (or $2x/(W-1) - 1$ for `True`), and use `padding_mode="border"` to get the clamp. Then multiply by the $[-1, H] \times [-1, W]$ mask and sum or average over the $(g_h, g_w)$ axes. `grid_sample` needs one grid per input image, so group the boxes by image, or index `input[batch_idx]` at a cost of $K \cdot C \cdot H \cdot W$ memory. A fixed `sampling_ratio` lets all boxes share one grid shape and batch cleanly.
5. **Why is RoIAlign's backward nondeterministic on GPU, and what does torchvision do about it?** Overlapping samples of overlapping boxes scatter into the same input pixels with `atomicAdd`, and floating-point addition order varies between runs. When `torch.use_deterministic_algorithms(True)` is set on CUDA, torchvision's `roi_align` switches to a pure-PyTorch, `torch.compile`d implementation (`_roi_align`) whose backward is deterministic.
6. **Does `aligned=True` vs `False` change accuracy?** The Detectron2 docs say the misalignment does not make a difference to model performance when RoIAlign is followed by conv layers, which can learn to compensate for the half-pixel shift. But `aligned=True` is the geometrically correct pixel model, and the setting has to match between training and inference (pretrained torchvision detectors use `False`). The same half-pixel question appears in `F.interpolate`'s `align_corners`.